In [ ]:
import torch
import numpy as np
a = torch.tensor([1,2,3,4,5,6,7])
b = torch.arange(0,9,1)
c = torch.linspace(0,2,5)
d = torch.zeros(3,4) 
e = torch.ones(3,4) 
f = torch.randn(3,4)
g = torch.randint(0,10,(2,3))
print(a)
np_array = np.array([1.0,2.0,3.0])
t_from_np = torch.from_numpy(np_array)
back_to_np = a.numpy()
print(t_from_np)
x = torch.tensor([1.0,2.0,3.0])
y = torch.tensor([4.0,5.0,6.0])
print(x@y)
m = torch.arange(12).reshape(3,4)
print(m)



tensor([1, 2, 3, 4, 5, 6, 7])
tensor([1., 2., 3.], dtype=torch.float64)
tensor(32.)
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [28]:
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
    x_gpu = torch.randn(3,4,device=device)
    print(x_gpu.device)
else:
    device = torch.device('cpu')
    x_cpu = torch.randn(3,4,device=device)
    print(x_cpu.device)
x = torch.tensor(3.0,requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()
print(x.item())
print(y.item())
print(x.grad.item())
x = torch.tensor(3.14,requires_grad=True)
print(x.detach())

cuda:0
3.0
16.0
8.0
tensor(3.1400)


In [31]:
import torch
x = torch.tensor(2.0,requires_grad=True)
f = x ** 3 + 2 * x ** 2 - 5 * x + 3
f.backward()
print(f.item())
print(x.grad.item())
a = torch.tensor(3.0,requires_grad=True)
b = torch.tensor(2.0,requires_grad=True)
g = a**2 * b + b**3
g.backward()
print(g.item())
print(a.grad.item())
print(b.grad.item())

9.0
15.0
26.0
12.0
21.0


In [12]:
import sys
import timeit
print(sys.getsizeof([]))
print(sys.getsizeof(()))
data_list = list(range(1000))
data_tuple = tuple(range(1000))
print(sys.getsizeof(data_list))
print(sys.getsizeof(data_tuple))
%timeit list(range(1000))
%timeit tuple(range(1000))
a = (1,2,3)
b = (1,2,3)
print(f'{a is b}')
c = [1,2,3]
d = [1,2,3]
print(c is d)

56
40
8056
8040
4.57 μs ± 91.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
4.99 μs ± 96.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
False
False


In [ ]:
class RingBuffer:
    def __init__(self,capacity:int):
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self._capacity = capacity
        self._buffer = [None] * capacity
        self._head = 0
        self._size = 0
    def append(self,item):
        self._buffer[self._head] = item
        self._head = (self._head + 1) % self._capacity
        self._size = min(self._size+1,self._capacity)
    def get_all(self) -> list:
        if self._size < self._capacity:
            return self._buffer[:self._size]
        start = self._head
        return self._buffer[start:]+self._buffer[:start]
    def __len__(self):
        return self._size
    def __repr__(self):
        return f"RingBuffer(capacity={self._capacity}, items={self.get_all()})"

In [15]:
"""
迁移学习实战：冻结预训练骨干网络，仅训练新增分类头
=====================================================
  - 骨干网络：ResNet-18（ImageNet 预训练）
  - 数据集 1：MNIST   (28×28, 灰度, 10类)
  - 数据集 2：CIFAR-10 (32×32, RGB,   10类)
"""

from sklearn import linear_model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time
import copy


# ============================================================
# 第一部分：构建冻结骨干 + 新分类头的模型
# ============================================================

def build_transfer_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    基于 ResNet-18 预训练权重，构建迁移学习模型：
      1. 冻结所有骨干网络参数（conv, bn, 原始 fc 全部冻结）
      2. 替换 fc 层为一个小型分类网络
    """
    # ---------- 加载预训练 ResNet-18 ----------
    weights = models.ResNet18_Weights.IMAGENET1K_V1  # 最新的 V2 权重也可
    model = models.resnet18(weights=weights)

    # ---------- 冻结骨干网络所有参数 ----------
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        print(f"[INFO] 骨干网络已冻结，共 {sum(p.numel() for p in model.parameters())} 个参数被锁定")

    # ---------- 获取骨干输出特征维度 ----------
    num_features = model.fc.in_features  # ResNet-18: 512

    # ---------- 替换原始 fc 为小型分类网络 ----------
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(num_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )

    # ---------- 统计可训练参数 ----------
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[INFO] 可训练参数: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.2f}%)")

    return model


# ============================================================
# 第二部分：数据加载与预处理
# ============================================================

def get_mnist_dataloaders(batch_size: int = 128, data_dir: str = "./data"):
    """
    MNIST 数据加载器
    关键点：
      - 原始 28×28 灰度图 → 缩放到 224×224 以适配 ResNet 输入
      - 灰度图(1通道) → 复制为3通道(RGB) 以适配预训练权重
      - 使用 ImageNet 均值/标准差进行归一化
    """
    # ImageNet 归一化参数（必须与预训练模型一致）
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    transform_train = transforms.Compose([
        transforms.Resize(256),              # 短边缩放到 256
        transforms.RandomResizedCrop(224),   # 随机裁剪到 224×224（数据增强）
        transforms.Grayscale(num_output_channels=3),  # 1通道 → 3通道
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    transform_val = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),          # 中心裁剪（验证时不用随机增强）
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    train_dataset = datasets.MNIST(
        root=data_dir, train=True, download=True, transform=transform_train
    )
    val_dataset = datasets.MNIST(
        root=data_dir, train=False, download=True, transform=transform_val
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)

    print(f"[MNIST] 训练集: {len(train_dataset):,} 张 | 验证集: {len(val_dataset):,} 张")
    return train_loader, val_loader
# ============================================================
# 第三部分：训练与评估引擎
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def run_training(model, train_loader, val_loader, device,
                 epochs=10, lr=1e-3, dataset_name="Dataset"):
    """
    训练循环：仅更新分类头参数，骨干网络保持冻结
    """
    # 只优化 requires_grad=True 的参数（即新分类头）
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    print(f"\n{'='*60}")
    print(f" 开始训练 [{dataset_name}] — 共 {epochs} 个 Epoch")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        start_time = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        elapsed = time.time() - start_time
        lr_now = optimizer.param_groups[0]['lr']

        print(f"  Epoch [{epoch:02d}/{epochs}]  "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
              f"LR: {lr_now:.6f} | {elapsed:.1f}s")

        # 保存最优模型
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    # 加载最优权重
    model.load_state_dict(best_model_wts)
    print(f"\n✅ [{dataset_name}] 训练完成！最佳验证准确率: {best_acc:.4f}")
    return model


# ============================================================
# 第四部分：主流程
# ============================================================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] 使用设备: {device}\n")

    # ==============================
    # 实验一：MNIST
    # ==============================
    print("▸" * 30 + " 实验一：MNIST " + "◂" * 30)
    mnist_train_loader, mnist_val_loader = get_mnist_dataloaders(batch_size=128)

    mnist_model = build_transfer_model(num_classes=10, freeze_backbone=True)
    mnist_model = mnist_model.to(device)

    mnist_model = run_training(
        model=mnist_model,
        train_loader=mnist_train_loader,
        val_loader=mnist_val_loader,
        device=device,
        epochs=5,        # MNIST 简单，5 个 epoch 即可
        lr=1e-3,
        dataset_name="MNIST"
    )

    torch.save(mnist_model.state_dict(), "resnet18_mnist_transfer.pth")
if __name__ == "__main__":
    main()

[INFO] 使用设备: cuda

▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸▸ 实验一：MNIST ◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂◂
[MNIST] 训练集: 60,000 张 | 验证集: 10,000 张
[INFO] 骨干网络已冻结，共 11689512 个参数被锁定
[INFO] 可训练参数: 133,898 / 11,310,410 (1.18%)

 开始训练 [MNIST] — 共 5 个 Epoch
  Epoch [01/5]  Train Loss: 1.4521  Acc: 0.5054 | Val Loss: 0.4356  Acc: 0.8881 | LR: 0.000905 | 61.2s
  Epoch [02/5]  Train Loss: 1.2768  Acc: 0.5609 | Val Loss: 0.3602  Acc: 0.8989 | LR: 0.000655 | 63.3s
  Epoch [03/5]  Train Loss: 1.2286  Acc: 0.5796 | Val Loss: 0.3422  Acc: 0.9022 | LR: 0.000345 | 63.3s
  Epoch [04/5]  Train Loss: 1.1994  Acc: 0.5866 | Val Loss: 0.3254  Acc: 0.9136 | LR: 0.000095 | 61.8s
  Epoch [05/5]  Train Loss: 1.1869  Acc: 0.5951 | Val Loss: 0.3144  Acc: 0.9153 | LR: 0.000000 | 61.8s

✅ [MNIST] 训练完成！最佳验证准确率: 0.9153
